# ACCT 3343 — Chapter 3: STA, SNOA, and PMAN Rankings

Today’s workflow: **company data → ask Kai for one step → paste → run → inspect**.

We use accounting-quality measures as **screening signals**. A high value is more concerning, but it does not prove manipulation or fraud.

The main 75-minute exercise ends after the demeaned two-measure ranking in Section 8. Sections 9–14 are a fully implemented optional extension.


## 1. Load the familiar Chapter 2 data

Press **Run**. The classroom dataset loads automatically. We immediately preserve the complete history before Chapter 2 narrows the rows and columns.


In [3]:
import math
from statistics import NormalDist

import numpy as np
import pandas as pd

EXPECTED_UNIVERSE_COUNT = 50
MIN_SECTOR_OBSERVATIONS = 3
DATA_URL = "https://raw.githubusercontent.com/x2003xuy/acct3343-public/main/data/acct3343_financial_data_50.csv"
VALIDATION_SNAPSHOT_SHA256 = "b48ba2400340a2447e781abf82333e993aaf45b55ea15ad9138a0e2b196e1813"

df = pd.read_csv(DATA_URL)
full_history_df = df.copy(deep=True)

print("Data loaded from:", DATA_URL)
print("Rows:", len(df), "| Companies:", df["ticker"].nunique())
df.head()


Data loaded from: https://raw.githubusercontent.com/x2003xuy/acct3343-public/main/data/acct3343_financial_data_50.csv
Rows: 248 | Companies: 50
  ticker fiscal_period_end  fiscal_year company_name      sector              industry exchange currency  current_share_price  current_market_cap  current_enterprise_value  current_shares_outstanding  current_shares_short  current_short_ratio  current_short_percent_of_float  current_insider_ownership_percent  current_institutional_ownership_percent  current_book_value_per_share  current_forward_eps  total_revenue  cost_of_revenue  gross_profit          ebit  operating_income        ebitda  normalized_income  pretax_income  tax_provision  tax_rate_for_calcs    net_income  net_income_continuing_operations  selling_general_and_administration  interest_expense  basic_average_shares  diluted_average_shares  basic_eps  diluted_eps  total_assets  current_assets  cash_and_cash_equivalents  cash_and_short_term_investments  accounts_receivable     invent

## 2. Keep each company’s latest fiscal period

This follows the Chapter 2 preparation. The latest period is selected **before** checking Chapter 3 inputs, so a company is never moved to an older year merely to improve coverage.


In [5]:
df["fiscal_period_end"] = pd.to_datetime(df["fiscal_period_end"], errors="coerce")
full_history_df["fiscal_period_end"] = pd.to_datetime(
    full_history_df["fiscal_period_end"], errors="coerce"
)

numeric_columns = [
    "gross_profit", "total_assets", "stockholders_equity",
    "ordinary_shares_number", "fiscal_period_end_close",
]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

duplicate_keys = full_history_df.duplicated(
    ["ticker", "fiscal_period_end"], keep=False
)
if duplicate_keys.any():
    display(full_history_df.loc[duplicate_keys].sort_values(["ticker", "fiscal_period_end"]))
    raise ValueError("Duplicate firm-period records require investigation.")

df = (
    df.sort_values(["ticker", "fiscal_period_end"], ascending=[True, False])
      .drop_duplicates("ticker", keep="first")
      .copy()
)

df["market_equity"] = df["ordinary_shares_number"] * df["fiscal_period_end_close"]

columns_to_keep = [
    "ticker", "company_name", "sector", "industry",
    "fiscal_period_end", "fiscal_year", "gross_profit", "total_assets",
    "stockholders_equity", "ordinary_shares_number",
    "fiscal_period_end_close", "market_equity",
]
df = df[columns_to_keep]

assert df["ticker"].nunique() == len(df)
assert len(df) == EXPECTED_UNIVERSE_COUNT

print("Companies:", df["ticker"].nunique())
df[["ticker", "fiscal_period_end", "sector"]].head()


Companies: 50
   ticker fiscal_period_end      sector
0    AAPL        2025-09-30  Technology
5    ABBV        2025-12-31  Healthcare
10    ABT        2025-12-31  Healthcare
15   ADBE        2025-11-30  Technology
20    AEP        2025-12-31   Utilities


## 3. Restore the accounting inputs needed for Chapter 3

Chapter 2 kept a narrow set of columns. We merge the **same latest firm-period keys** back to the preserved full-history table. `full_history_df` remains multi-year; `master_df` has one latest row per firm.


In [7]:
chapter3_numeric_columns = [
    "net_income", "operating_cash_flow", "total_assets",
    "cash_and_cash_equivalents", "current_debt_and_capital_lease_obligation",
    "long_term_debt_and_capital_lease_obligation", "total_debt",
    "minority_interest", "preferred_stock", "common_stock_equity",
    "stockholders_equity", "total_liabilities_net_minority_interest",
    "total_revenue", "cost_of_revenue", "accounts_receivable",
    "current_assets", "net_property_plant_equipment",
    "depreciation_and_amortization", "selling_general_and_administration",
    "current_liabilities", "current_debt", "taxes_payable", "long_term_debt",
]

for column in chapter3_numeric_columns:
    full_history_df[column] = pd.to_numeric(full_history_df[column], errors="coerce")

latest_keys = df[["ticker", "fiscal_period_end"]].copy()
master_df = latest_keys.merge(
    full_history_df,
    on=["ticker", "fiscal_period_end"],
    how="left",
    validate="one_to_one",
)
master_df = master_df.sort_values("ticker").reset_index(drop=True)

assert len(master_df) == EXPECTED_UNIVERSE_COUNT
assert master_df[["ticker", "fiscal_period_end"]].duplicated().sum() == 0

master_df[["ticker", "fiscal_period_end", "net_income", "operating_cash_flow", "total_assets"]].head()


  ticker fiscal_period_end    net_income  operating_cash_flow  total_assets
0   AAPL        2025-09-30  1.120100e+11         1.114820e+11  3.592410e+11
1   ABBV        2025-12-31  4.226000e+09         1.903000e+10  1.339600e+11
2    ABT        2025-12-31  6.524000e+09         9.566000e+09  8.671300e+10
3   ADBE        2025-11-30  7.130000e+09         1.003100e+10  2.949600e+10
4    AEP        2025-12-31  3.580000e+09         6.944000e+09  1.144600e+11


## 4. Scaled total accruals (STA)

The lecture’s simplified measure uses the same-period year-end asset denominator:

\[
STA = \frac{Net\ Income - Cash\ Flow\ from\ Operations}{Total\ Assets}
\]

Yahoo’s `operating_cash_flow` is mapped to cash flow from operations. `total_assets` is the year-end balance for the same fiscal period. A valid positive denominator is required. The signed value is retained; higher STA is more concerning.

Ask Kai:

> **master_df: create STA = (net_income - operating_cash_flow) / total_assets. Require positive total_assets and leave invalid rows missing. Code only.**


In [9]:
# Paste Kai's code here, then press Run


### Reference solution — STA


In [11]:
valid_assets = master_df["total_assets"].gt(0)
valid_sta_inputs = master_df[["net_income", "operating_cash_flow"]].notna().all(axis=1)

master_df["STA"] = (
    (master_df["net_income"] - master_df["operating_cash_flow"])
    / master_df["total_assets"]
).where(valid_assets & valid_sta_inputs)

master_df["STA_Missing_Reason"] = ""
master_df.loc[~valid_assets, "STA_Missing_Reason"] = "Total assets missing or non-positive"
master_df.loc[
    valid_assets & ~valid_sta_inputs, "STA_Missing_Reason"
] = "Missing net income or operating cash flow"

master_df[
    ["ticker", "fiscal_period_end", "net_income", "operating_cash_flow", "total_assets", "STA"]
].sort_values(["STA", "ticker"], ascending=[False, True], na_position="last").head(10)


   ticker fiscal_period_end    net_income  operating_cash_flow  total_assets       STA
33   NVDA        2026-01-31  1.200670e+11         1.027180e+11  2.068030e+11  0.083891
24     KO        2025-12-31  1.310700e+10         7.408000e+09  1.048160e+11  0.054371
23    JPM        2025-12-31  5.704800e+10        -1.477820e+11  4.424900e+12  0.046290
18     GS        2025-12-31  1.717600e+10        -4.515400e+10  1.809320e+12  0.034449
29     MS        2025-12-31  1.686100e+10        -1.788900e+10  1.420270e+12  0.024467
47    WFC        2025-12-31  2.133800e+10        -1.900100e+10  2.148631e+12  0.018774
28    MRK        2025-12-31  1.825400e+10         1.647200e+10  1.368660e+11  0.013020
22    JNJ        2025-12-31  2.680400e+10         2.453000e+10  1.992100e+11  0.011415
7     BLK        2025-12-31  5.553000e+09         3.927000e+09  1.699980e+11  0.009565
32    NKE        2026-05-31  3.108000e+09         2.868000e+09  3.841000e+10  0.006248


## 5. Scaled net operating assets (SNOA)

The lecture builds SNOA from total assets:

\[
Operating\ Assets = Total\ Assets - Cash\ and\ Cash\ Equivalents
\]

\[
Operating\ Liabilities = Total\ Assets - Short\text{-}Term\ Debt - Long\text{-}Term\ Debt - Minority\ Interest - Preferred\ Stock - Common\ Equity
\]

\[
SNOA = \frac{Operating\ Assets - Operating\ Liabilities}{Total\ Assets}
\]

We use Yahoo’s debt lines **including capital-lease obligations**, exactly as listed in the lecture. We do not add current debt to a total-debt field. Missing minority interest is reconstructed only when the balance-sheet identity supports it. Missing preferred stock becomes zero only under the lecture’s structural-absence convention and only when the core balance-sheet/equity lines are present.

Ask Kai:

> **master_df: prepare the slide inputs for SNOA using cash_and_cash_equivalents, debt-and-capital-lease fields, minority_interest, preferred_stock, and common_stock_equity. Show any reconstructions. Code only.**


In [13]:
# Paste Kai's code here, then press Run


### Reference solution — careful SNOA input preparation


In [15]:
# Short-term debt including capital leases.
master_df["SNOA_Current_Debt_Lease_Used"] = master_df[
    "current_debt_and_capital_lease_obligation"
]
master_df["SNOA_Current_Debt_Lease_Source"] = (
    "Yahoo Current Debt And Capital Lease Obligation"
)

current_debt_residual = (
    master_df["total_debt"]
    - master_df["long_term_debt_and_capital_lease_obligation"]
)
debt_tolerance = master_df["total_assets"].abs() * 1e-9
reconstruct_current_debt = (
    master_df["SNOA_Current_Debt_Lease_Used"].isna()
    & current_debt_residual.notna()
    & current_debt_residual.ge(-debt_tolerance)
)
master_df.loc[
    reconstruct_current_debt, "SNOA_Current_Debt_Lease_Used"
] = current_debt_residual.loc[reconstruct_current_debt].clip(lower=0)
master_df.loc[
    reconstruct_current_debt, "SNOA_Current_Debt_Lease_Source"
] = "Total Debt minus Long Term Debt And Capital Lease Obligation"
master_df.loc[
    master_df["SNOA_Current_Debt_Lease_Used"].isna(),
    "SNOA_Current_Debt_Lease_Source",
] = "Unavailable"

# Minority interest: use the reported line first. Reconstruct only from a
# balance-sheet identity. Small negative residuals within 0.01% of assets are
# treated as rounding; other unexplained gaps stay missing.
master_df["SNOA_Minority_Interest_Used"] = master_df["minority_interest"]
master_df["SNOA_Minority_Interest_Source"] = "Yahoo Minority Interest"
minority_residual = (
    master_df["total_assets"]
    - master_df["total_liabilities_net_minority_interest"]
    - master_df["stockholders_equity"]
)
minority_tolerance = master_df["total_assets"].abs() * 1e-4
reconstruct_minority = (
    master_df["SNOA_Minority_Interest_Used"].isna()
    & minority_residual.notna()
    & minority_residual.ge(-minority_tolerance)
)
minority_reconstructed = minority_residual.where(
    minority_residual.abs().gt(minority_tolerance), 0.0
)
master_df.loc[
    reconstruct_minority, "SNOA_Minority_Interest_Used"
] = minority_reconstructed.loc[reconstruct_minority]
master_df.loc[
    reconstruct_minority, "SNOA_Minority_Interest_Source"
] = "Balance-sheet identity reconstruction"
master_df.loc[
    master_df["SNOA_Minority_Interest_Used"].isna(),
    "SNOA_Minority_Interest_Source",
] = "Unavailable"

# Preferred stock: the lecture permits zero for structural absence. This is
# not a blanket fill; it is applied only when the core equity lines are present.
master_df["SNOA_Preferred_Stock_Used"] = master_df["preferred_stock"]
master_df["SNOA_Preferred_Stock_Source"] = "Yahoo Preferred Stock"
preferred_structural_absence = (
    master_df["SNOA_Preferred_Stock_Used"].isna()
    & master_df[["total_assets", "common_stock_equity", "stockholders_equity"]]
      .notna().all(axis=1)
)
master_df.loc[
    preferred_structural_absence, "SNOA_Preferred_Stock_Used"
] = 0.0
master_df.loc[
    preferred_structural_absence, "SNOA_Preferred_Stock_Source"
] = "Zero: documented structural absence"
master_df.loc[
    master_df["SNOA_Preferred_Stock_Used"].isna(),
    "SNOA_Preferred_Stock_Source",
] = "Unavailable"

master_df[[
    "ticker", "SNOA_Current_Debt_Lease_Used", "SNOA_Current_Debt_Lease_Source",
    "SNOA_Minority_Interest_Used", "SNOA_Minority_Interest_Source",
    "SNOA_Preferred_Stock_Used", "SNOA_Preferred_Stock_Source",
]].head(10)


  ticker  SNOA_Current_Debt_Lease_Used                                SNOA_Current_Debt_Lease_Source  SNOA_Minority_Interest_Used          SNOA_Minority_Interest_Source  SNOA_Preferred_Stock_Used          SNOA_Preferred_Stock_Source
0   AAPL                  2.032900e+10               Yahoo Current Debt And Capital Lease Obligation                 0.000000e+00  Balance-sheet identity reconstruction               0.000000e+00  Zero: documented structural absence
1   ABBV                  8.555000e+09               Yahoo Current Debt And Capital Lease Obligation                 4.200000e+07                Yahoo Minority Interest               0.000000e+00  Zero: documented structural absence
2    ABT                  3.033000e+09               Yahoo Current Debt And Capital Lease Obligation                 6.410000e+08                Yahoo Minority Interest               0.000000e+00                Yahoo Preferred Stock
3   ADBE                  7.700000e+07               Yahoo Current D

Ask Kai:

> **master_df: calculate Operating_Assets, Operating_Liabilities, and SNOA from the prepared slide inputs. Require positive total_assets. Higher SNOA is more concerning. Code only.**


In [17]:
# Paste Kai's code here, then press Run


### Reference solution — SNOA


In [19]:
master_df["Operating_Assets"] = (
    master_df["total_assets"] - master_df["cash_and_cash_equivalents"]
)
master_df["Operating_Liabilities"] = (
    master_df["total_assets"]
    - master_df["SNOA_Current_Debt_Lease_Used"]
    - master_df["long_term_debt_and_capital_lease_obligation"]
    - master_df["SNOA_Minority_Interest_Used"]
    - master_df["SNOA_Preferred_Stock_Used"]
    - master_df["common_stock_equity"]
)

snoa_inputs = [
    "cash_and_cash_equivalents", "SNOA_Current_Debt_Lease_Used",
    "long_term_debt_and_capital_lease_obligation",
    "SNOA_Minority_Interest_Used", "SNOA_Preferred_Stock_Used",
    "common_stock_equity",
]
valid_snoa = valid_assets & master_df[snoa_inputs].notna().all(axis=1)
master_df["SNOA"] = (
    (master_df["Operating_Assets"] - master_df["Operating_Liabilities"])
    / master_df["total_assets"]
).where(valid_snoa)

master_df["SNOA_Missing_Reason"] = ""
master_df.loc[~valid_assets, "SNOA_Missing_Reason"] = "Total assets missing or non-positive"
missing_snoa_inputs = valid_assets & master_df[snoa_inputs].isna().any(axis=1)
master_df.loc[missing_snoa_inputs, "SNOA_Missing_Reason"] = (
    master_df.loc[missing_snoa_inputs, snoa_inputs]
    .isna()
    .apply(lambda row: "Missing SNOA input(s): " + ", ".join(row.index[row]), axis=1)
)

master_df["Financial_Company_Flag"] = master_df["sector"].eq("Financial Services")
master_df["Model_Applicability_Note"] = ""
master_df.loc[master_df["Financial_Company_Flag"], "Model_Applicability_Note"] = (
    "Financial-company statements can make industrial-company accrual screens less comparable."
)

master_df[[
    "ticker", "fiscal_period_end", "Operating_Assets",
    "Operating_Liabilities", "total_assets", "SNOA", "SNOA_Missing_Reason",
]].sort_values(["SNOA", "ticker"], ascending=[False, True], na_position="last").head(10)


   ticker fiscal_period_end  Operating_Assets  Operating_Liabilities  total_assets      SNOA SNOA_Missing_Reason
26    MCD        2025-12-31      5.874100e+10           6.491000e+09  5.951500e+10  0.877930                    
14     DE        2025-10-31      9.772000e+10           1.573900e+10  1.059960e+11  0.773435                    
43    TMO        2025-12-31      1.004910e+11           1.742200e+10  1.103430e+11  0.752825                    
31    NEE        2025-12-31      2.099090e+11           5.062300e+10  2.127210e+11  0.748802                    
15    DUK        2025-12-31      1.954910e+11           5.184800e+10  1.957360e+11  0.733861                    
19     HD        2026-01-31      1.037060e+11           2.693200e+10  1.050950e+11  0.730520                    
36    PFE        2025-12-31      2.070180e+11           5.742400e+10  2.081600e+11  0.718649                    
45    UNP        2025-12-31      6.843200e+10           1.840900e+10  6.969800e+10  0.717711    

## 6. Rank STA and SNOA, then combine them

Convention for all Chapter 3 rankings: **Rank 1 = most concerning**. Higher STA and SNOA receive lower rank numbers. Ties use `method="min"`. Missing values remain unranked.

Standalone component ranks may use different samples. For the combined score, both components are re-ranked inside the same complete-case sample.

Ask Kai:

> **master_df: rank STA and SNOA descending with method='min'. Re-rank both within firms valid on both, sum to RAW2_Score, then rank the sum ascending into RAW2_Rank. Code only.**


In [21]:
# Paste Kai's code here, then press Run


### Reference solution — raw two-measure ranking


In [23]:
master_df["STA_Standalone_Rank"] = master_df["STA"].rank(
    ascending=False, method="min"
)
master_df["SNOA_Standalone_Rank"] = master_df["SNOA"].rank(
    ascending=False, method="min"
)

master_df["RAW2_Eligible"] = master_df[["STA", "SNOA"]].notna().all(axis=1)
master_df["RAW2_STA_Rank"] = np.nan
master_df["RAW2_SNOA_Rank"] = np.nan
master_df.loc[master_df["RAW2_Eligible"], "RAW2_STA_Rank"] = (
    master_df.loc[master_df["RAW2_Eligible"], "STA"]
    .rank(ascending=False, method="min")
)
master_df.loc[master_df["RAW2_Eligible"], "RAW2_SNOA_Rank"] = (
    master_df.loc[master_df["RAW2_Eligible"], "SNOA"]
    .rank(ascending=False, method="min")
)
master_df["RAW2_Score"] = np.nan
master_df.loc[master_df["RAW2_Eligible"], "RAW2_Score"] = (
    master_df.loc[
        master_df["RAW2_Eligible"], ["RAW2_STA_Rank", "RAW2_SNOA_Rank"]
    ].sum(axis=1, min_count=2)
)
master_df["RAW2_Rank"] = np.nan
master_df.loc[master_df["RAW2_Eligible"], "RAW2_Rank"] = (
    master_df.loc[master_df["RAW2_Eligible"], "RAW2_Score"]
    .rank(ascending=True, method="min")
)

raw2_table = master_df[[
    "ticker", "fiscal_period_end", "STA", "RAW2_STA_Rank",
    "SNOA", "RAW2_SNOA_Rank", "RAW2_Score", "RAW2_Rank",
]].sort_values(["RAW2_Rank", "ticker"], na_position="last")

print("RAW2 eligible firms:", int(master_df["RAW2_Eligible"].sum()))
raw2_table.head(15)


RAW2 eligible firms: 49
   ticker fiscal_period_end       STA  RAW2_STA_Rank      SNOA  RAW2_SNOA_Rank  RAW2_Score  RAW2_Rank
24     KO        2025-12-31  0.054371            1.0  0.663038            16.0        17.0        1.0
43    TMO        2025-12-31 -0.010096           14.0  0.752825             3.0        17.0        1.0
14     DE        2025-10-31 -0.022944           20.0  0.773435             2.0        22.0        3.0
19     HD        2026-01-31 -0.020638           17.0  0.730520             6.0        23.0        4.0
36    PFE        2025-12-31 -0.018894           16.0  0.718649             7.0        23.0        4.0
28    MRK        2025-12-31  0.013020            6.0  0.638844            19.0        25.0        6.0
31    NEE        2025-12-31 -0.026561           22.0  0.748802             4.0        26.0        7.0
26    MCD        2025-12-31 -0.033403           27.0  0.877930             1.0        28.0        8.0
45    UNP        2025-12-31 -0.030876           26.0  0.71

## 7. Compare each measure with its broad-sector mean

Ratios can have different typical levels across industries. Subtracting the broad `sector` mean compares a firm with broad peers rather than treating every raw ratio as directly comparable.

For each measure separately, the sector mean uses all valid latest-period observations in the original 50-firm universe. At least **three valid observations per sector per measure** are required. A positive deviation means above the sector mean and remains more concerning.

This teaching extension is useful but imperfect. Mean subtraction does not equalize variances, remove outlier sensitivity, or make a three-firm mean highly reliable.

Ask Kai:

> **master_df: for STA and SNOA separately, calculate broad-sector valid counts and means. Create STA_Demeaned and SNOA_Demeaned only when the sector has at least 3 valid values. Do not pool missing sectors. Code only.**


In [25]:
# Paste Kai's code here, then press Run


### Reference solution — sector means and deviations


In [27]:
for measure in ["STA", "SNOA"]:
    master_df[f"{measure}_Sector_Valid_N"] = (
        master_df.groupby("sector", dropna=True)[measure].transform("count").astype("Int64")
    )
    master_df[f"{measure}_Sector_Mean"] = (
        master_df.groupby("sector", dropna=True)[measure].transform("mean")
    )
    can_demean = (
        master_df["sector"].notna()
        & master_df[measure].notna()
        & master_df[f"{measure}_Sector_Valid_N"].ge(MIN_SECTOR_OBSERVATIONS)
    )
    master_df[f"{measure}_Demeaned"] = (
        master_df[measure] - master_df[f"{measure}_Sector_Mean"]
    ).where(can_demean)
    master_df[f"{measure}_Demean_Flag"] = "Eligible"
    master_df.loc[master_df["sector"].isna(), f"{measure}_Demean_Flag"] = (
        "Missing sector classification"
    )
    master_df.loc[master_df[measure].isna(), f"{measure}_Demean_Flag"] = (
        f"Missing {measure}"
    )
    master_df.loc[
        master_df[measure].notna()
        & master_df["sector"].notna()
        & master_df[f"{measure}_Sector_Valid_N"].lt(MIN_SECTOR_OBSERVATIONS),
        f"{measure}_Demean_Flag",
    ] = f"Fewer than {MIN_SECTOR_OBSERVATIONS} valid sector observations"

sector_summary_2 = (
    master_df.groupby("sector", dropna=False)
    .agg(
        Firms=("ticker", "size"),
        STA_Valid=("STA", "count"),
        STA_Mean=("STA", "mean"),
        SNOA_Valid=("SNOA", "count"),
        SNOA_Mean=("SNOA", "mean"),
    )
)
sector_summary_2


                    Firms  STA_Valid  STA_Mean  SNOA_Valid  SNOA_Mean
sector                                                               
Consumer Cyclical       7          7 -0.044508           7   0.608402
Consumer Defensive      6          6 -0.035520           6   0.540113
Energy                  5          5 -0.073933           5   0.672233
Financial Services      6          6  0.023132           6   0.199114
Healthcare              7          7 -0.024975           7   0.610742
Industrials             7          7 -0.023725           7   0.554742
Technology              8          8 -0.028250           7   0.544757
Utilities               4          4 -0.032158           4   0.727430


In [28]:
master_df[[
    "ticker", "sector", "STA", "STA_Sector_Valid_N", "STA_Sector_Mean", "STA_Demeaned",
    "SNOA", "SNOA_Sector_Valid_N", "SNOA_Sector_Mean", "SNOA_Demeaned",
]].sort_values(["sector", "ticker"]).head(15)


   ticker              sector       STA  STA_Sector_Valid_N  STA_Sector_Mean  STA_Demeaned      SNOA  SNOA_Sector_Valid_N  SNOA_Sector_Mean  SNOA_Demeaned
5    AMZN   Consumer Cyclical -0.075600                   7        -0.044508     -0.031092  0.583395                    7          0.608402      -0.025006
19     HD   Consumer Cyclical -0.020638                   7        -0.044508      0.023869  0.730520                    7          0.608402       0.122118
25    LOW   Consumer Cyclical -0.059286                   7        -0.044508     -0.014779  0.623855                    7          0.608402       0.015453
26    MCD   Consumer Cyclical -0.033403                   7        -0.044508      0.011104  0.877930                    7          0.608402       0.269528
32    NKE   Consumer Cyclical  0.006248                   7        -0.044508      0.050756  0.477350                    7          0.608402      -0.131052
39   SBUX   Consumer Cyclical -0.090291                   7        -0.

## 8. Re-rank the two sector-demeaned measures

The common eligible sample is formed first. Both component ranks are recalculated inside that same sample, summed, and then re-ranked. Rank 1 remains most concerning.

Ask Kai:

> **master_df: require both demeaned measures, rank each descending inside that sample, sum ranks to DM2_Score, and rank the sum ascending into DM2_Rank. Code only.**


In [30]:
# Paste Kai's code here, then press Run


### Reference solution — demeaned two-measure ranking


In [32]:
master_df["DM2_Eligible"] = master_df[["STA_Demeaned", "SNOA_Demeaned"]].notna().all(axis=1)
for measure in ["STA_Demeaned", "SNOA_Demeaned"]:
    rank_column = f"DM2_{measure}_Rank"
    master_df[rank_column] = np.nan
    master_df.loc[master_df["DM2_Eligible"], rank_column] = (
        master_df.loc[master_df["DM2_Eligible"], measure]
        .rank(ascending=False, method="min")
    )

master_df["DM2_Score"] = np.nan
master_df.loc[master_df["DM2_Eligible"], "DM2_Score"] = master_df.loc[
    master_df["DM2_Eligible"],
    ["DM2_STA_Demeaned_Rank", "DM2_SNOA_Demeaned_Rank"],
].sum(axis=1, min_count=2)
master_df["DM2_Rank"] = np.nan
master_df.loc[master_df["DM2_Eligible"], "DM2_Rank"] = (
    master_df.loc[master_df["DM2_Eligible"], "DM2_Score"]
    .rank(ascending=True, method="min")
)

dm2_table = master_df[[
    "ticker", "sector", "STA_Demeaned", "DM2_STA_Demeaned_Rank",
    "SNOA_Demeaned", "DM2_SNOA_Demeaned_Rank", "DM2_Score", "DM2_Rank",
]].sort_values(["DM2_Rank", "ticker"], na_position="last")

print("DM2 eligible firms:", int(master_df["DM2_Eligible"].sum()))
dm2_table.head(15)


DM2 eligible firms: 49
   ticker              sector  STA_Demeaned  DM2_STA_Demeaned_Rank  SNOA_Demeaned  DM2_SNOA_Demeaned_Rank  DM2_Score  DM2_Rank
24     KO  Consumer Defensive      0.089891                    1.0       0.122925                     6.0        7.0       1.0
19     HD   Consumer Cyclical      0.023869                    7.0       0.122118                     7.0       14.0       2.0
26    MCD   Consumer Cyclical      0.011104                   15.0       0.269528                     1.0       16.0       3.0
43    TMO          Healthcare      0.014880                   12.0       0.142083                     5.0       17.0       4.0
28    MRK          Healthcare      0.037996                    3.0       0.028102                    17.0       20.0       5.0
36    PFE          Healthcare      0.006081                   18.0       0.107907                     8.0       26.0       6.0
37     PG  Consumer Defensive      0.007777                   16.0       0.087411       

# OPTIONAL EXTENSION — Historical PROBM and PMAN

The class can stop here with a complete STA/SNOA exercise. The remaining sections are implemented for an extended class or instructor demonstration.

The original Beneish eight-variable probit specification is used. The Chapter 3 reading gives the coefficients and converts the score with the **standard normal CDF**, not a logistic function:

\[
PROBM = -4.84 + 0.920DSRI + 0.528GMI + 0.404AQI + 0.892SGI + 0.115DEPI - 0.172SGAI + 4.679TATA - 0.327LVGI
\]

\[
PMAN = \Phi(PROBM)
\]

Source: Messod D. Beneish, “The Detection of Earnings Manipulation,” *Financial Analysts Journal* 55(5), 1999, Table 2 definitions and Table 3 unweighted-probit coefficients, pp. 27–29. The course reading log covers *Quantitative Value*, Chapter 3, pp. 62–79.

The original Beneish estimation excludes financial institutions. We flag those firms as not applicable rather than manufacturing industrial-company inputs.


## 9. Reload and align the original multi-year data

Lags are created only within ticker after sorting. A valid comparison must be the preceding annual fiscal period: the gap must be 330–400 days and the fiscal year must advance by one. This accepts ordinary 52/53-week reporting but rejects a two-year gap.


In [35]:
historical_df = pd.read_csv(DATA_URL)
historical_df["fiscal_period_end"] = pd.to_datetime(
    historical_df["fiscal_period_end"], errors="coerce"
)
historical_df["fiscal_year"] = pd.to_numeric(
    historical_df["fiscal_year"], errors="coerce"
).astype("Int64")
for column in chapter3_numeric_columns:
    historical_df[column] = pd.to_numeric(historical_df[column], errors="coerce")

if historical_df.duplicated(["ticker", "fiscal_period_end"], keep=False).any():
    raise ValueError("Duplicate historical firm-period keys require investigation.")

historical_df = historical_df.sort_values(["ticker", "fiscal_period_end"]).copy()

lag_columns = [
    "fiscal_period_end", "fiscal_year", "total_revenue", "cost_of_revenue",
    "accounts_receivable", "current_assets", "net_property_plant_equipment",
    "total_assets", "depreciation_and_amortization",
    "selling_general_and_administration", "current_liabilities", "long_term_debt",
    "current_debt", "taxes_payable", "cash_and_cash_equivalents",
]
for column in lag_columns:
    historical_df[f"Prior_{column}"] = (
        historical_df.groupby("ticker", sort=False)[column].shift(1)
    )

historical_df["Prior_Gap_Days"] = (
    historical_df["fiscal_period_end"] - historical_df["Prior_fiscal_period_end"]
).dt.days.astype("Int64")
historical_df["Valid_Preceding_Annual_Period"] = (
    historical_df["Prior_Gap_Days"].between(330, 400, inclusive="both")
    & (historical_df["fiscal_year"] - historical_df["Prior_fiscal_year"]).eq(1)
)

def safe_divide(numerator, denominator, positive_denominator=False):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    valid = numerator.notna() & denominator.notna()
    valid &= denominator.gt(0) if positive_denominator else denominator.ne(0)
    result = pd.Series(np.nan, index=numerator.index, dtype="float64")
    result.loc[valid] = numerator.loc[valid] / denominator.loc[valid]
    result.loc[~np.isfinite(result)] = np.nan
    return result

historical_df[[
    "ticker", "Prior_fiscal_period_end", "fiscal_period_end",
    "Prior_Gap_Days", "Valid_Preceding_Annual_Period",
]].tail(10)


    ticker Prior_fiscal_period_end fiscal_period_end  Prior_Gap_Days  Valid_Preceding_Annual_Period
242    WMT                     NaT        2022-01-31            <NA>                           <NA>
241    WMT              2022-01-31        2023-01-31             365                           True
240    WMT              2023-01-31        2024-01-31             365                           True
239    WMT              2024-01-31        2025-01-31             366                           True
238    WMT              2025-01-31        2026-01-31             365                           True
247    XOM                     NaT        2021-12-31            <NA>                           <NA>
246    XOM              2021-12-31        2022-12-31             365                           True
245    XOM              2022-12-31        2023-12-31             365                           True
244    XOM              2023-12-31        2024-12-31             366                           True


## 10. Calculate the eight Beneish components

### DSRI — Days’ sales in receivables index

`(accounts_receivable / total_revenue)_t ÷ (accounts_receivable / total_revenue)_{t-1}`. Higher values can indicate that receivables are growing faster than sales.


In [37]:
receivable_rate_t = safe_divide(
    historical_df["accounts_receivable"], historical_df["total_revenue"], True
)
receivable_rate_prior = safe_divide(
    historical_df["Prior_accounts_receivable"], historical_df["Prior_total_revenue"], True
)
historical_df["DSRI"] = safe_divide(receivable_rate_t, receivable_rate_prior)


### GMI — Gross margin index

`gross margin_{t-1} ÷ gross margin_t`, where gross margin is `(sales − cost of revenue) / sales`. A value above 1 means the margin deteriorated. The ratio direction is prior year over current year.


In [39]:
gross_margin_t = safe_divide(
    historical_df["total_revenue"] - historical_df["cost_of_revenue"],
    historical_df["total_revenue"], True,
)
gross_margin_prior = safe_divide(
    historical_df["Prior_total_revenue"] - historical_df["Prior_cost_of_revenue"],
    historical_df["Prior_total_revenue"], True,
)
historical_df["GMI"] = safe_divide(gross_margin_prior, gross_margin_t)


### AQI — Asset quality index

Asset quality is `1 − (current assets + net PP&E) / total assets`. AQI is current asset quality divided by prior asset quality. Following the original paper, noncurrent investments remain in the residual rather than being subtracted as a separate investment line.


In [41]:
asset_quality_t = 1 - safe_divide(
    historical_df["current_assets"] + historical_df["net_property_plant_equipment"],
    historical_df["total_assets"], True,
)
asset_quality_prior = 1 - safe_divide(
    historical_df["Prior_current_assets"] + historical_df["Prior_net_property_plant_equipment"],
    historical_df["Prior_total_assets"], True,
)
historical_df["AQI"] = safe_divide(asset_quality_t, asset_quality_prior)


### SGI — Sales growth index

`total_revenue_t ÷ total_revenue_{t-1}`.


In [43]:
historical_df["SGI"] = safe_divide(
    historical_df["total_revenue"], historical_df["Prior_total_revenue"], True
)


### DEPI — Depreciation index

The depreciation rate is `depreciation / (depreciation + net PP&E)`. DEPI is the prior rate divided by the current rate. Yahoo provides combined depreciation and amortization, not the paper’s depreciation net of separately reported intangible amortization. We use the combined Yahoo line as a material, labelled approximation and leave missing values missing.


In [45]:
depreciation_rate_t = safe_divide(
    historical_df["depreciation_and_amortization"],
    historical_df["depreciation_and_amortization"]
    + historical_df["net_property_plant_equipment"],
    True,
)
depreciation_rate_prior = safe_divide(
    historical_df["Prior_depreciation_and_amortization"],
    historical_df["Prior_depreciation_and_amortization"]
    + historical_df["Prior_net_property_plant_equipment"],
    True,
)
historical_df["DEPI"] = safe_divide(depreciation_rate_prior, depreciation_rate_t)


### SGAI — SG&A expense index

`(SG&A / sales)_t ÷ (SG&A / sales)_{t-1}`. The collection pipeline tries Yahoo’s combined SG&A line first and then General and Administrative Expense. Because row-level source-label provenance was not retained, a populated value can be a partial proxy for issuers that report only G&A. This limitation is material.


In [47]:
sga_rate_t = safe_divide(
    historical_df["selling_general_and_administration"],
    historical_df["total_revenue"], True,
)
sga_rate_prior = safe_divide(
    historical_df["Prior_selling_general_and_administration"],
    historical_df["Prior_total_revenue"], True,
)
historical_df["SGAI"] = safe_divide(sga_rate_t, sga_rate_prior)


### LVGI — Leverage index

The original paper defines leverage as `(long-term debt + current liabilities) / total assets`. LVGI is the current leverage ratio divided by the prior ratio. We do not substitute `total_debt` or add current debt again because current liabilities already contain the current portion.


In [49]:
leverage_t = safe_divide(
    historical_df["long_term_debt"] + historical_df["current_liabilities"],
    historical_df["total_assets"], True,
)
leverage_prior = safe_divide(
    historical_df["Prior_long_term_debt"] + historical_df["Prior_current_liabilities"],
    historical_df["Prior_total_assets"], True,
)
historical_df["LVGI"] = safe_divide(leverage_t, leverage_prior)


### TATA — Total accruals to total assets

The published balance-sheet formula is:

`[Δ current assets − Δ cash − (Δ current liabilities − Δ current debt − Δ taxes payable) − depreciation and amortization] / current total assets`

This is **not** the simplified STA measure. It requires balance-sheet changes and the separate current-debt and taxes-payable lines. Missing values stay missing; we do not substitute zero or a neutral ratio.


In [51]:
total_accruals = (
    (historical_df["current_assets"] - historical_df["Prior_current_assets"])
    - (historical_df["cash_and_cash_equivalents"] - historical_df["Prior_cash_and_cash_equivalents"])
    - (
        (historical_df["current_liabilities"] - historical_df["Prior_current_liabilities"])
        - (historical_df["current_debt"] - historical_df["Prior_current_debt"])
        - (historical_df["taxes_payable"] - historical_df["Prior_taxes_payable"])
    )
    - historical_df["depreciation_and_amortization"]
)
historical_df["TATA"] = safe_divide(
    total_accruals, historical_df["total_assets"], True
)

historical_df[[
    "ticker", "fiscal_period_end", "DSRI", "GMI", "AQI", "SGI",
    "DEPI", "SGAI", "TATA", "LVGI",
]].tail(10)


    ticker fiscal_period_end      DSRI       GMI       AQI       SGI      DEPI      SGAI      TATA      LVGI
242    WMT        2022-01-31       NaN       NaN       NaN       NaN       NaN       NaN       NaN       NaN
241    WMT        2023-01-31       NaN       NaN       NaN       NaN       NaN       NaN       NaN       NaN
240    WMT        2024-01-31  1.045769  0.990362  0.901231  1.060260  1.008599  0.971585 -0.050507  0.976455
239    WMT        2025-01-31  1.079317  0.980859  0.892250  1.050700  0.981046  1.016516 -0.046349  0.978528
238    WMT        2026-01-31  1.069465  0.996966  0.942121  1.047252  1.023968       NaN -0.059374  1.001582
247    XOM        2021-12-31       NaN       NaN       NaN       NaN       NaN       NaN       NaN       NaN
246    XOM        2022-12-31       NaN       NaN       NaN       NaN       NaN       NaN       NaN       NaN
245    XOM        2023-12-31  1.098744  1.028347  0.951699  0.839523  1.199546  1.170385 -0.048900  0.912981
244    XOM        2

## 11. Calculate PROBM and convert it to PMAN

All eight components, a valid preceding annual period, and model applicability are required. Undefined ratios are not replaced with 1. PMAN is stored on the 0–1 scale.

Ask Kai:

> **historical_df: calculate the published eight-variable PROBM score and PMAN = standard normal CDF(PROBM). Require every component. Do not use a logistic transformation. Code only.**


In [53]:
# Paste Kai's code here, then press Run


### Reference solution — PROBM and PMAN


In [55]:
beneish_components = ["DSRI", "GMI", "AQI", "SGI", "DEPI", "SGAI", "TATA", "LVGI"]
historical_df["PMAN_Model_Applicable"] = ~historical_df["sector"].eq("Financial Services")
historical_df["PROBM_Eligible"] = (
    historical_df["Valid_Preceding_Annual_Period"]
    & historical_df["PMAN_Model_Applicable"]
    & historical_df[beneish_components].notna().all(axis=1)
)

historical_df["PROBM"] = np.nan
eligible = historical_df["PROBM_Eligible"]
historical_df.loc[eligible, "PROBM"] = (
    -4.84
    + 0.920 * historical_df.loc[eligible, "DSRI"]
    + 0.528 * historical_df.loc[eligible, "GMI"]
    + 0.404 * historical_df.loc[eligible, "AQI"]
    + 0.892 * historical_df.loc[eligible, "SGI"]
    + 0.115 * historical_df.loc[eligible, "DEPI"]
    - 0.172 * historical_df.loc[eligible, "SGAI"]
    + 4.679 * historical_df.loc[eligible, "TATA"]
    - 0.327 * historical_df.loc[eligible, "LVGI"]
)

normal = NormalDist()
historical_df["PMAN"] = historical_df["PROBM"].map(
    lambda value: normal.cdf(value) if pd.notna(value) else np.nan
)

historical_df["PMAN_Missing_Reason"] = ""
historical_df.loc[
    ~historical_df["PMAN_Model_Applicable"], "PMAN_Missing_Reason"
] = "Original Beneish estimation excluded financial institutions"
historical_df.loc[
    historical_df["PMAN_Model_Applicable"]
    & ~historical_df["Valid_Preceding_Annual_Period"],
    "PMAN_Missing_Reason",
] = "No valid preceding annual fiscal period"

undefined_components = (
    historical_df["PMAN_Model_Applicable"]
    & historical_df["Valid_Preceding_Annual_Period"]
    & historical_df[beneish_components].isna().any(axis=1)
)
historical_df.loc[undefined_components, "PMAN_Missing_Reason"] = (
    historical_df.loc[undefined_components, beneish_components]
    .isna()
    .apply(lambda row: "Undefined component(s): " + ", ".join(row.index[row]), axis=1)
)

historical_df.loc[historical_df["PMAN"].notna(), [
    "ticker", "fiscal_period_end", *beneish_components, "PROBM", "PMAN",
]].tail(10)


    ticker fiscal_period_end      DSRI       GMI       AQI       SGI      DEPI      SGAI      TATA      LVGI     PROBM      PMAN
194   SBUX        2024-09-30  1.019398  1.019776  0.968873  1.005576  1.031887  1.027857 -0.069459  0.959953 -2.772342  0.002783
193   SBUX        2025-09-30  1.023943  1.178615  1.021850  1.027869  0.900583  1.009091 -0.063517  1.037201 -2.652335  0.003997
200    SLB        2023-12-31  0.978837  0.927580  0.954853  1.179559  1.045572  0.820718 -0.045520  0.964090 -2.560948  0.005219
199    SLB        2024-12-31  0.936346  0.963500  0.979040  1.095186  0.959067  0.965765 -0.024052  0.963719 -2.580891  0.004927
198    SLB        2025-12-31  1.102282  1.128102  1.060939  0.983990  0.983943  0.897486 -0.033043  0.915405 -2.419084  0.007780
240    WMT        2024-01-31  1.045769  0.990362  0.901231  1.060260  1.008599  0.971585 -0.050507  0.976455 -2.651881  0.004002
239    WMT        2025-01-31  1.079317  0.980859  0.892250  1.050700  0.981046  1.016516 -0.04634

## 12. Merge same-period PMAN into the 50-firm master table

The merge uses both `ticker` and `fiscal_period_end`, validates one-to-one keys, keeps all 50 master rows, and never backfills an older PMAN.


In [57]:
pman_columns = [
    "ticker", "fiscal_period_end", "Prior_fiscal_period_end", "Prior_Gap_Days",
    "Valid_Preceding_Annual_Period", "PMAN_Model_Applicable",
    *beneish_components, "PROBM_Eligible", "PROBM", "PMAN", "PMAN_Missing_Reason",
]
# Make this merge cell safe to re-run after an instructor demonstration.
existing_pman_columns = [
    column for column in pman_columns
    if column not in ["ticker", "fiscal_period_end"] and column in master_df.columns
]
master_df = master_df.drop(columns=existing_pman_columns)
latest_pman = latest_keys.merge(
    historical_df[pman_columns],
    on=["ticker", "fiscal_period_end"],
    how="left",
    validate="one_to_one",
)
master_df = master_df.merge(
    latest_pman,
    on=["ticker", "fiscal_period_end"],
    how="left",
    validate="one_to_one",
)

assert len(master_df) == EXPECTED_UNIVERSE_COUNT
assert master_df[["ticker", "fiscal_period_end"]].duplicated().sum() == 0

master_df[[
    "ticker", "fiscal_period_end", "Prior_fiscal_period_end", "PROBM", "PMAN", "PMAN_Missing_Reason",
]].sort_values(["PMAN", "ticker"], ascending=[False, True], na_position="last").head(15)


   ticker fiscal_period_end Prior_fiscal_period_end     PROBM      PMAN PMAN_Missing_Reason
25    LOW        2026-01-31              2025-01-31  8.903542  1.000000                    
32    NKE        2026-05-31              2025-05-31 -2.204383  0.013749                    
24     KO        2025-12-31              2024-12-31 -2.207557  0.013638                    
19     HD        2026-01-31              2025-01-31 -2.347885  0.009440                    
12   CSCO        2026-07-31              2025-07-31 -2.378245  0.008698                    
28    MRK        2025-12-31              2024-12-31 -2.385013  0.008539                    
40    SLB        2025-12-31              2024-12-31 -2.419084  0.007780                    
2     ABT        2025-12-31              2024-12-31 -2.470145  0.006753                    
22    JNJ        2025-12-31              2024-12-31 -2.479871  0.006572                    
30   MSFT        2026-06-30              2025-06-30 -2.555163  0.005307         

## 13. Build the raw and demeaned three-measure rankings

PMAN is the third signal. PROBM and PMAN are not counted separately. All three component ranks are recalculated inside the common eligible sample and receive equal weight through their rank sum.


In [59]:
master_df["RAW3_Eligible"] = master_df[["STA", "SNOA", "PMAN"]].notna().all(axis=1)
for measure in ["STA", "SNOA", "PMAN"]:
    rank_column = f"RAW3_{measure}_Rank"
    master_df[rank_column] = np.nan
    master_df.loc[master_df["RAW3_Eligible"], rank_column] = (
        master_df.loc[master_df["RAW3_Eligible"], measure]
        .rank(ascending=False, method="min")
    )

master_df["RAW3_Score"] = np.nan
master_df.loc[master_df["RAW3_Eligible"], "RAW3_Score"] = master_df.loc[
    master_df["RAW3_Eligible"],
    ["RAW3_STA_Rank", "RAW3_SNOA_Rank", "RAW3_PMAN_Rank"],
].sum(axis=1, min_count=3)
master_df["RAW3_Rank"] = np.nan
master_df.loc[master_df["RAW3_Eligible"], "RAW3_Rank"] = (
    master_df.loc[master_df["RAW3_Eligible"], "RAW3_Score"]
    .rank(ascending=True, method="min")
)

raw3_table = master_df[[
    "ticker", "STA", "RAW3_STA_Rank", "SNOA", "RAW3_SNOA_Rank",
    "PMAN", "RAW3_PMAN_Rank", "RAW3_Score", "RAW3_Rank",
]].sort_values(["RAW3_Rank", "ticker"], na_position="last")

print("RAW3 eligible firms:", int(master_df["RAW3_Eligible"].sum()))
raw3_table.head(15)


RAW3 eligible firms: 16
   ticker       STA  RAW3_STA_Rank      SNOA  RAW3_SNOA_Rank      PMAN  RAW3_PMAN_Rank  RAW3_Score  RAW3_Rank
24     KO  0.054371            1.0  0.663038             6.0  0.013638             3.0        10.0        1.0
19     HD -0.020638            8.0  0.730520             1.0  0.009440             4.0        13.0        2.0
28    MRK  0.013020            2.0  0.638844             8.0  0.008539             6.0        16.0        3.0
32    NKE  0.006248            4.0  0.477350            15.0  0.013749             2.0        21.0        4.0
2     ABT -0.035081           10.0  0.670130             4.0  0.006753             8.0        22.0        5.0
12   CSCO -0.007020            6.0  0.560025            11.0  0.008698             5.0        22.0        5.0
36    PFE -0.018894            7.0  0.718649             2.0  0.003504            14.0        23.0        7.0
22    JNJ  0.011415            3.0  0.551017            12.0  0.006572             9.0        24

PMAN is demeaned **after** the probability conversion. `PMAN_Demeaned` is a deviation from a sector mean, not a probability. Earlier STA/SNOA deviations remain unchanged.


In [61]:
master_df["PMAN_Sector_Valid_N"] = (
    master_df.groupby("sector", dropna=True)["PMAN"].transform("count").astype("Int64")
)
master_df["PMAN_Sector_Mean"] = (
    master_df.groupby("sector", dropna=True)["PMAN"].transform("mean")
)
can_demean_pman = (
    master_df["sector"].notna()
    & master_df["PMAN"].notna()
    & master_df["PMAN_Sector_Valid_N"].ge(MIN_SECTOR_OBSERVATIONS)
)
master_df["PMAN_Demeaned"] = (
    master_df["PMAN"] - master_df["PMAN_Sector_Mean"]
).where(can_demean_pman)
master_df["PMAN_Demean_Flag"] = "Eligible"
master_df.loc[master_df["sector"].isna(), "PMAN_Demean_Flag"] = "Missing sector classification"
master_df.loc[master_df["PMAN"].isna(), "PMAN_Demean_Flag"] = "Missing PMAN"
master_df.loc[
    master_df["PMAN"].notna()
    & master_df["sector"].notna()
    & master_df["PMAN_Sector_Valid_N"].lt(MIN_SECTOR_OBSERVATIONS),
    "PMAN_Demean_Flag",
] = "Fewer than 3 valid sector PMAN observations"

master_df["DM3_Eligible"] = master_df[[
    "STA_Demeaned", "SNOA_Demeaned", "PMAN_Demeaned"
]].notna().all(axis=1)
for measure in ["STA_Demeaned", "SNOA_Demeaned", "PMAN_Demeaned"]:
    rank_column = f"DM3_{measure}_Rank"
    master_df[rank_column] = np.nan
    master_df.loc[master_df["DM3_Eligible"], rank_column] = (
        master_df.loc[master_df["DM3_Eligible"], measure]
        .rank(ascending=False, method="min")
    )

master_df["DM3_Score"] = np.nan
master_df.loc[master_df["DM3_Eligible"], "DM3_Score"] = master_df.loc[
    master_df["DM3_Eligible"],
    ["DM3_STA_Demeaned_Rank", "DM3_SNOA_Demeaned_Rank", "DM3_PMAN_Demeaned_Rank"],
].sum(axis=1, min_count=3)
master_df["DM3_Rank"] = np.nan
master_df.loc[master_df["DM3_Eligible"], "DM3_Rank"] = (
    master_df.loc[master_df["DM3_Eligible"], "DM3_Score"]
    .rank(ascending=True, method="min")
)

dm3_table = master_df[[
    "ticker", "sector", "STA_Demeaned", "DM3_STA_Demeaned_Rank",
    "SNOA_Demeaned", "DM3_SNOA_Demeaned_Rank",
    "PMAN_Demeaned", "DM3_PMAN_Demeaned_Rank", "DM3_Score", "DM3_Rank",
]].sort_values(["DM3_Rank", "ticker"], na_position="last")

print("DM3 eligible firms:", int(master_df["DM3_Eligible"].sum()))
dm3_table.head(15)


DM3 eligible firms: 14
   ticker             sector  STA_Demeaned  DM3_STA_Demeaned_Rank  SNOA_Demeaned  DM3_SNOA_Demeaned_Rank  PMAN_Demeaned  DM3_PMAN_Demeaned_Rank  DM3_Score  DM3_Rank
28    MRK         Healthcare      0.037996                    2.0       0.028102                     5.0       0.002197                     4.0       11.0       1.0
12   CSCO         Technology      0.021230                    7.0       0.015268                     8.0       0.002654                     3.0       18.0       2.0
19     HD  Consumer Cyclical      0.023869                    5.0       0.122118                     1.0      -0.247356                    13.0       19.0       3.0
2     ABT         Healthcare     -0.010106                   11.0       0.059388                     4.0       0.000411                     5.0       20.0       4.0
22    JNJ         Healthcare      0.036391                    3.0      -0.059726                    11.0       0.000230                     6.0       20

## 14. Final comparison and interpretation

Differences between RAW2, DM2, RAW3, and DM3 can reflect both the added signal and changes in the eligible sample.


In [63]:
comparison_table = master_df[[
    "ticker", "company_name", "sector", "fiscal_period_end",
    "STA", "SNOA", "PMAN",
    "RAW2_Eligible", "RAW2_Rank", "DM2_Eligible", "DM2_Rank",
    "RAW3_Eligible", "RAW3_Rank", "DM3_Eligible", "DM3_Rank",
    "STA_Demean_Flag", "SNOA_Demean_Flag", "PMAN_Demean_Flag",
    "PMAN_Missing_Reason", "Model_Applicability_Note",
]].sort_values(["RAW2_Rank", "ticker"], na_position="last")

sample_sizes = pd.Series({
    "RAW2": int(master_df["RAW2_Eligible"].sum()),
    "DM2": int(master_df["DM2_Eligible"].sum()),
    "RAW3": int(master_df["RAW3_Eligible"].sum()),
    "DM3": int(master_df["DM3_Eligible"].sum()),
}, name="Eligible firms")

display(sample_sizes.to_frame())
comparison_table.head(20)


      Eligible firms
RAW2              49
DM2               49
RAW3              16
DM3               14
   ticker                                 company_name              sector fiscal_period_end       STA      SNOA      PMAN  RAW2_Eligible  RAW2_Rank  DM2_Eligible  DM2_Rank  RAW3_Eligible  RAW3_Rank  DM3_Eligible  DM3_Rank STA_Demean_Flag SNOA_Demean_Flag                             PMAN_Demean_Flag                                          PMAN_Missing_Reason                                                                   Model_Applicability_Note
24     KO                        The Coca-Cola Company  Consumer Defensive        2025-12-31  0.054371  0.663038  0.013638           True        1.0          True       1.0           True        1.0         False       NaN        Eligible         Eligible  Fewer than 3 valid sector PMAN observations                                                                                                                                              

### Discussion

- Which firms move the most after sector demeaning?
- Which differences come from relative sector position, and which come from a changed eligible sample?
- Why can an extreme PMAN dominate attention even though the ranking gives each component equal rank weight?
- Which missing-data reasons reflect unavailable inputs, and which reflect model applicability?
- Why should these rankings trigger investigation rather than a conclusion about manipulation?


## 15. Concise validation summary and export


In [66]:
# Data and numerical assertions. Detailed synthetic fixtures and independent
# worked examples are in the repository validation report and test suite.
assert len(master_df) == EXPECTED_UNIVERSE_COUNT
assert master_df["ticker"].nunique() == EXPECTED_UNIVERSE_COUNT
assert master_df[["ticker", "fiscal_period_end"]].duplicated().sum() == 0
numeric_values = master_df.select_dtypes(include="number").to_numpy(dtype=float)
assert np.isfinite(numeric_values[~np.isnan(numeric_values)]).all()
assert master_df.loc[~master_df["RAW2_Eligible"], "RAW2_Score"].isna().all()
assert master_df.loc[~master_df["RAW3_Eligible"], "RAW3_Score"].isna().all()
assert master_df.loc[~master_df["DM3_Eligible"], "DM3_Score"].isna().all()
assert master_df["PMAN"].dropna().between(0, 1, inclusive="both").all()

for measure in ["STA", "SNOA", "PMAN"]:
    valid_deviation = master_df[f"{measure}_Demeaned"].notna()
    if valid_deviation.any():
        sector_zero_checks = (
            master_df.loc[valid_deviation]
            .groupby("sector")[f"{measure}_Demeaned"]
            .mean()
            .abs()
        )
        assert (sector_zero_checks < 1e-12).all()

export_columns = [
    "ticker", "company_name", "sector", "industry", "fiscal_period_end", "fiscal_year",
    "STA", "STA_Missing_Reason", "SNOA", "SNOA_Missing_Reason",
    "Operating_Assets", "Operating_Liabilities",
    "SNOA_Current_Debt_Lease_Used", "SNOA_Minority_Interest_Used", "SNOA_Preferred_Stock_Used",
    "SNOA_Current_Debt_Lease_Source", "SNOA_Minority_Interest_Source",
    "SNOA_Preferred_Stock_Source", "Financial_Company_Flag", "Model_Applicability_Note",
    "STA_Sector_Valid_N", "STA_Sector_Mean", "STA_Demeaned", "STA_Demean_Flag",
    "SNOA_Sector_Valid_N", "SNOA_Sector_Mean", "SNOA_Demeaned", "SNOA_Demean_Flag",
    "Prior_fiscal_period_end", "Prior_Gap_Days", "Valid_Preceding_Annual_Period",
    *beneish_components, "PROBM", "PMAN", "PMAN_Missing_Reason",
    "PMAN_Sector_Valid_N", "PMAN_Sector_Mean", "PMAN_Demeaned", "PMAN_Demean_Flag",
    "STA_Standalone_Rank", "SNOA_Standalone_Rank",
    "RAW2_Eligible", "RAW2_STA_Rank", "RAW2_SNOA_Rank", "RAW2_Score", "RAW2_Rank",
    "DM2_Eligible", "DM2_STA_Demeaned_Rank", "DM2_SNOA_Demeaned_Rank", "DM2_Score", "DM2_Rank",
    "RAW3_Eligible", "RAW3_STA_Rank", "RAW3_SNOA_Rank", "RAW3_PMAN_Rank", "RAW3_Score", "RAW3_Rank",
    "DM3_Eligible", "DM3_STA_Demeaned_Rank", "DM3_SNOA_Demeaned_Rank",
    "DM3_PMAN_Demeaned_Rank", "DM3_Score", "DM3_Rank",
]
final_results = master_df[export_columns].sort_values("ticker").copy()
final_results.to_csv("ACCT3343_Chapter3_final_results.csv", index=False)

print("Validation checks passed.")
print("Exported rows:", len(final_results))
sample_sizes.to_frame()


Validation checks passed.
Exported rows: 50
      Eligible firms
RAW2              49
DM2               49
RAW3              16
DM3               14
